# Introduction to Autodifferentiation with PyTorch

In PyTorch, **autodifferentiation** (or **autograd**) lets us compute derivatives automatically.

This is useful because many machine learning and inverse problems require gradients of a loss function with respect to model parameters.

The basic idea is:

- define tensors
- tell PyTorch which ones need gradients with `requires_grad=True`
- perform calculations normally
- call `.backward()` on a scalar result
- read the gradients from `.grad`

PyTorch records operations in a **computational graph** during the forward pass and then applies the chain rule during the backward pass. Only **leaf tensors** with `requires_grad=True` store gradients in their `.grad` field. 

In [1]:
import torch

print("PyTorch version:", torch.__version__)

PyTorch version: 2.4.1.post300


## 1. A simple scalar example

Let

$$
y = x^2 + 3x + 1
$$

Then

$$
\frac{dy}{dx} = 2x + 3
$$

At $x=2$, the exact derivative is

$$
\frac{dy}{dx} = 2(2) + 3 = 7
$$

We will let PyTorch compute this automatically.

In [2]:
x = torch.tensor(2.0, requires_grad=True)

y = x**2 + 3*x + 1
print("y =", y.item())

y.backward()   # compute dy/dx
print("dy/dx =", x.grad.item())

y = 11.0
dy/dx = 7.0


The result should be `7.0`, which matches the analytical derivative.

Notice:

- `x` is a tensor with `requires_grad=True`
- `y` is built from `x`
- calling `y.backward()` computes the derivative of `y` with respect to `x`
- the result is stored in `x.grad`

## 2. A slightly more interesting function

Now consider

$$
f(x) = \sin(x)e^x
$$

Using the product rule,

$$
f'(x) = e^x \sin(x) + e^x \cos(x)
       = e^x(\sin(x)+\cos(x))
$$

We compare the PyTorch derivative with the analytical result.

In [3]:
x = torch.tensor(1.0, requires_grad=True)

f = torch.sin(x) * torch.exp(x)
f.backward()

autograd_value = x.grad.item()
exact_value = (torch.exp(torch.tensor(1.0)) * 
               (torch.sin(torch.tensor(1.0)) + torch.cos(torch.tensor(1.0)))).item()

print("Autograd derivative :", autograd_value)
print("Analytical derivative:", exact_value)
print("Absolute difference  :", abs(autograd_value - exact_value))

Autograd derivative : 3.756049156188965
Analytical derivative: 3.7560489177703857
Absolute difference  : 2.384185791015625e-07


## 3. Vector example and partial derivatives

Autograd also works for multivariable functions.

Let

$$
Q(a,b) = 3a^3 - b^2
$$

Then

$$
\frac{\partial Q}{\partial a} = 9a^2,
\qquad
\frac{\partial Q}{\partial b} = -2b
$$

For $a=2$ and $b=4$:

$$
\frac{\partial Q}{\partial a} = 36,
\qquad
\frac{\partial Q}{\partial b} = -8
$$

In [4]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)

Q = 3*a**3 - b**2
Q.backward()

print("Q =", Q.item())
print("dQ/da =", a.grad.item())
print("dQ/db =", b.grad.item())

Q = 8.0
dQ/da = 36.0
dQ/db = -8.0


This is an example of computing **partial derivatives** of a scalar output with respect to several inputs.

In machine learning, the scalar output is usually the **loss function**, and the inputs are the model parameters.

## 4. Gradients accumulate unless we clear them

A common beginner mistake is forgetting that PyTorch **accumulates** gradients in `.grad`.

If `.backward()` is called twice, the new gradient is added to the old one unless we reset it.

In [5]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x**2
y1.backward()
print("After first backward, grad =", x.grad.item())

y2 = 3*x
y2.backward()
print("After second backward, grad =", x.grad.item(), "(accumulated)")

After first backward, grad = 4.0
After second backward, grad = 7.0 (accumulated)


x = torch.tensor(2.0, requires_grad=True)

y1 = x**2
y1.backward()
print("First grad =", x.grad.item())

x.grad.zero_()   # clear stored gradient

y2 = 3*x
y2.backward()
print("After zeroing, new grad =", x.grad.item())

This is why, during training loops, we usually clear gradients before computing a new backward pass.

## 5. A tiny linear regression example

Now we use autodifferentiation in a way that resembles machine learning.

We want to fit a straight line

$$
\hat{y} = wx + b
$$

to a few data points.

We will:

1. define parameters $w$ and $b$
2. compute predictions
3. compute the mean squared error loss
4. call `.backward()` to get $\frac{\partial L}{\partial w}$ and $\frac{\partial L}{\partial b}$
5. update the parameters using gradient descent

In [6]:
# small synthetic dataset: y = 2x + 1 with slight noise
x_data = torch.tensor([[0.0], [1.0], [2.0], [3.0], [4.0]])
y_data = torch.tensor([[1.1], [2.9], [5.2], [7.1], [8.9]])

# parameters to learn
w = torch.tensor([[0.0]], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

learning_rate = 0.05

for epoch in range(101):
    # forward model
    y_pred = x_data @ w + b
    
    # mean squared error
    loss = ((y_pred - y_data)**2).mean()
    
    # backward
    loss.backward()
    
    # gradient descent step (no grad tracking during manual update)
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    
    # clear gradients
    w.grad.zero_()
    b.grad.zero_()
    
    if epoch % 20 == 0:
        print(f"epoch={epoch:3d}  loss={loss.item():.6f}  w={w.item():.4f}  b={b.item():.4f}")

epoch=  0  loss=33.256001  w=1.4040  b=0.5040
epoch= 20  loss=0.025351  w=2.0437  b=0.8983
epoch= 40  loss=0.017097  w=2.0148  b=0.9809
epoch= 60  loss=0.014641  w=1.9990  b=1.0259
epoch= 80  loss=0.013910  w=1.9903  b=1.0505
epoch=100  loss=0.013692  w=1.9856  b=1.0639


As the iterations progress, the parameters $w$ and $b$ should move toward values close to

$$
w \approx 2, \qquad b \approx 1
$$

The important point is that we never derived the gradients by hand inside the code. PyTorch computed them automatically from the loss function.

## 6. Doing the same thing with `nn.Linear`

PyTorch also provides modules such as `torch.nn.Linear`, standard losses such as `torch.nn.MSELoss`, and optimizers such as `torch.optim.SGD`.

This is the standard style used in many training scripts.

In [7]:
import torch.nn as nn

# reproducibility
torch.manual_seed(0)

x_data = torch.tensor([[0.0], [1.0], [2.0], [3.0], [4.0]])
y_data = torch.tensor([[1.1], [2.9], [5.2], [7.1], [8.9]])

model = nn.Linear(1, 1)      # y = wx + b
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(101):
    y_pred = model(x_data)
    loss = loss_fn(y_pred, y_data)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        w = model.weight.item()
        b = model.bias.item()
        print(f"epoch={epoch:3d}  loss={loss.item():.6f}  w={w:.4f}  b={b:.4f}")

epoch=  0  loss=28.330921  w=1.2937  b=0.9883
epoch= 20  loss=0.015466  w=1.9546  b=1.1524
epoch= 40  loss=0.014155  w=1.9661  b=1.1195
epoch= 60  loss=0.013765  w=1.9724  b=1.1015
epoch= 80  loss=0.013649  w=1.9759  b=1.0918
epoch=100  loss=0.013615  w=1.9778  b=1.0864


This version is shorter because:

- `nn.Linear(1,1)` creates the parameters automatically
- `nn.MSELoss()` computes the mean squared error
- `torch.optim.SGD(...)` updates the parameters
- `optimizer.zero_grad()` clears accumulated gradients

## 7. Key ideas to remember

- `requires_grad=True` tells PyTorch to track operations on a tensor
- the forward calculations define a computational graph
- `.backward()` applies the chain rule automatically
- gradients are stored in `.grad`
- gradients accumulate unless they are reset
- in learning problems, autodiff is used to compute derivatives of the loss with respect to model parameters

## 8. Suggested exercise for students

Modify the scalar example and compute the derivative of

$$
g(x) = x^3 - 4x + 5
$$

at $x=2$.

Then compare:

1. the analytical derivative
2. the PyTorch derivative
3. a finite-difference approximation

In [8]:
x = torch.tensor(2.0, requires_grad=True)
g = x**3 - 4*x + 5
g.backward()

autograd_grad = x.grad.item()
analytic_grad = 3*(2.0**2) - 4

# finite difference
h = 1e-5
x1 = 2.0 + h
x0 = 2.0
fd_grad = ((x1**3 - 4*x1 + 5) - (x0**3 - 4*x0 + 5)) / h

print("Autograd :", autograd_grad)
print("Analytic :", analytic_grad)
print("Finite diff:", fd_grad)

Autograd : 8.0
Analytic : 8.0
Finite diff: 8.00006000023501
